# 12 — Targeted Passage Search (Batch 03)

## Objective

Find candidate passages for the next ~40 labels, focusing on the categories that are missing or
weak in the current 39-passage dataset:

| Target | selection_method | Why |
|---|---|---|
| Music | targeted | Music-mediated processing has 0 labels |
| Poetry | targeted | Poetry-mediated processing has 0 labels |
| Walton's letters | targeted | Writing as narrative frame vs writing as processing |
| Writing | targeted | Scriptotherapy has 0 labels |
| Reading | targeted | Hard case: is *reading* (e.g. the creature reading Victor's journal) scriptotherapy or reconstruction? |
| Support / comfort | hard_negative | Tests the "support ≠ catharsis" rule |
| Reflection + despair | hard_negative | Tests the "attempt vs outcome" rule |
| Random | random | A few unbiased passages, so the batch isn't 100% cherry-picked |

**Important:** keyword search only *suggests* candidates. Many will turn out not to be therapeutic at all.
That's fine. Label them honestly as 0, because they become useful hard negatives.

## 1. Setup and load data

In [1]:
import re
import sys
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

IN_COLAB = "google.colab" in sys.modules
SEED = 42


def find_file(name):
    for p in [name, f"data/processed/{name}", f"../data/processed/{name}"]:
        if Path(p).exists():
            return p
    if IN_COLAB:
        from google.colab import files
        print(f"Please upload {name}")
        files.upload()
        return name
    raise FileNotFoundError(f"Could not find {name}. Put it next to this notebook.")


passages = pd.read_csv(find_file("frankenstein_passages.csv"))
labelled = pd.read_csv(find_file("frankenstein_training_dataset_v2.csv"))
labelled_ids = set(labelled["passage_id"])

print("All passages:", len(passages))
print("Already labelled:", len(labelled_ids))
print("Unlabelled:", len(passages) - len(labelled_ids))

Please upload frankenstein_passages.csv


Saving frankenstein_passages.csv to frankenstein_passages.csv
Please upload frankenstein_training_dataset_v2.csv


Saving frankenstein_training_dataset_v2.csv to frankenstein_training_dataset_v2.csv
All passages: 229
Already labelled: 39
Unlabelled: 190


## 2. Add chapter and position

The passages file has no chapter column, so we recover it from the "Letter N" / "Chapter N" markers in the text.
Each passage gets the last marker that appears in or before it (approximate, but good enough to navigate).

In [2]:
MARKER = re.compile(r"\b(Letter \d+|Chapter \d+)\b")

chapters, current = [], "Opening"
for text in passages["text"]:
    found = MARKER.findall(text)
    if found:
        current = found[-1]
    chapters.append(current)

passages["chapter"] = chapters
passages["position"] = (passages.index / (len(passages) - 1)).round(3)
passages["already_labelled"] = passages["passage_id"].isin(labelled_ids)
passages[["passage_id", "chapter", "position", "already_labelled"]].head(12)

,passage_id,chapter,position,already_labelled
0,1,Letter 1,0.000,False
1,2,Letter 1,0.004,False
2,3,Letter 1,0.009,False
3,4,Letter 2,0.013,False
4,5,Letter 2,0.018,False
5,6,Letter 2,0.022,False
6,7,Letter 2,0.026,False
7,8,Letter 3,0.031,False
8,9,Letter 4,0.035,False
9,10,Letter 4,0.039,True


## 3. Cue words for each target

- **Strong cues** count 2 points, **weak cues** count 1 point.
- `min_score` is the minimum a passage needs to count as a candidate.
- `require_strong` means at least one strong cue must appear.
- Edit these lists and re-run if you want to search differently.

In [3]:
TARGETS = {
    "music": {
        "selection_method": "targeted",
        "strong": [r"music\w*", r"melod\w*", r"songs?", r"sang", r"sing(?:s|ing|er|ers)?",
                   r"tunes?", r"guitar", r"flute", r"lute", r"harp"],
        "weak": [r"harmon\w*", r"instruments?"],
        "min_score": 2,
        "require_strong": True,      # "instruments" alone is usually Victor's lab equipment
    },
    "poetry": {
        "selection_method": "targeted",
        "strong": [r"poe(?:m|ms|try|t|ts|tic|tical)", r"verses?", r"rhym\w*", r"odes?", r"stanzas?",
                   r"wordsworth", r"coleridge", r"ancient mariner", r"milton", r"paradise lost"],
        "weak": [],
        "min_score": 2,
    },
    "walton_letters": {
        "selection_method": "targeted",
        "strong": [r"writ(?:e|es|ing|ten)", r"wrote", r"record\w*", r"journal"],
        "weak": [r"letters?", r"sister", r"margaret"],
        "min_score": 0,              # chosen by position (Letters 1-4), cues only rank them
        "chapter_prefix": "Letter",
    },
    "writing": {
        "selection_method": "targeted",
        "strong": [r"writ(?:e|es|ing|ten)", r"wrote", r"pen", r"journal", r"diar(?:y|ies)",
                   r"manuscripts?", r"notes", r"record\w*"],
        "weak": [r"letters?", r"papers"],
        "min_score": 3,
        "require_strong": True,      # a passage that only mentions "letters" isn't enough
    },
    "reading": {
        "selection_method": "targeted",
        "strong": [r"read(?:ing)?", r"books?", r"volumes?", r"werter", r"plutarch",
                   r"paradise lost", r"journal"],
        "weak": [r"papers", r"studied", r"study"],
        "min_score": 4,
        "require_strong": True,
    },
    "support": {
        "selection_method": "hard_negative",
        "strong": [r"consol\w*", r"comfort\w*", r"sooth\w*", r"sympath\w*"],
        "weak": [r"kind(?:ness|ly)?", r"tender\w*", r"affection\w*", r"embrac\w*", r"cheer\w*", r"gentle"],
        "min_score": 4,
    },
    "reflection_despair": {
        "selection_method": "hard_negative",
        # needs BOTH groups: reflection words AND despair words
        "strong": [r"remember\w*", r"recollect\w*", r"reflect\w*", r"memor(?:y|ies)", r"past"],
        "weak": [r"despair\w*", r"miser\w*", r"wretched\w*", r"anguish", r"remorse", r"hopeless\w*", r"agony"],
        "min_score": 4,
        "require_both": True,
    },
}

# How many NEW (unlabelled) passages to pick per target (~40 total)
QUOTA = {"music": 6, "poetry": 6, "walton_letters": 4, "writing": 6,
         "reading": 4, "support": 5, "reflection_despair": 5}
N_RANDOM = 4

compiled = {}
for t, cfg in TARGETS.items():
    compiled[t] = {
        "strong": re.compile(r"\b(" + "|".join(cfg["strong"]) + r")\b", re.I) if cfg["strong"] else None,
        "weak": re.compile(r"\b(" + "|".join(cfg["weak"]) + r")\b", re.I) if cfg["weak"] else None,
    }
print("Targets:", list(TARGETS))
print("Planned batch size:", sum(QUOTA.values()) + N_RANDOM)

Targets: ['music', 'poetry', 'walton_letters', 'writing', 'reading', 'support', 'reflection_despair']
Planned batch size: 40


## 4. Score every passage for every target

In [4]:
def score(text, chapter, target):
    cfg, pats = TARGETS[target], compiled[target]
    strong = [m.group(0).lower() for m in pats["strong"].finditer(text)] if pats["strong"] else []
    weak = [m.group(0).lower() for m in pats["weak"].finditer(text)] if pats["weak"] else []
    s = 2 * len(strong) + len(weak)
    ok = s >= cfg["min_score"]
    if cfg.get("require_strong"):
        ok = ok and len(strong) > 0
    if cfg.get("require_both"):
        ok = ok and len(strong) > 0 and len(weak) > 0
    if cfg.get("chapter_prefix"):
        ok = ok and chapter.startswith(cfg["chapter_prefix"])
    cues = ", ".join(sorted(set(strong + weak)))
    return s, ok, cues


rows = []
for _, p in passages.iterrows():
    for t in TARGETS:
        s, ok, cues = score(p["text"], p["chapter"], t)
        rows.append({"passage_id": p["passage_id"], "target": t, "score": s,
                     "eligible": ok, "matched_cues": cues})
scores = pd.DataFrame(rows)

# Overview: how many passages qualify for each target?
overview = scores[scores["eligible"]].merge(passages[["passage_id", "already_labelled"]], on="passage_id")
summary = overview.groupby("target")["already_labelled"].agg(eligible="size", already_labelled="sum")
summary["unlabelled"] = summary["eligible"] - summary["already_labelled"]
summary.reindex(list(TARGETS))

,eligible,already_labelled,unlabelled
target,,,
music,11,1,10
poetry,8,1,7
walton_letters,17,3,14
writing,14,1,13
reading,17,2,15
support,30,8,22
reflection_despair,47,17,30


### Already-labelled passages that match a target

Worth re-checking against guidelines v1.1. Some may need a `secondary_mechanism`
(for example, passage 214 matches writing cues).

In [5]:
relabel = overview[overview["already_labelled"]].merge(
    labelled[["passage_id", "therapeutic_label", "mechanism"]], on="passage_id")
relabel = relabel[relabel["target"].isin(["music", "poetry", "writing", "reading", "walton_letters"])]
relabel.sort_values(["target", "score"], ascending=[True, False])[
    ["passage_id", "target", "score", "matched_cues", "therapeutic_label", "mechanism"]]

,passage_id,target,score,matched_cues,therapeutic_label,mechanism
3,16,music,2,music,1.0,Narrative Reconstruction
23,153,poetry,4,"poetry, wordsworth",1.0,Emotional Catharsis
17,123,reading,21,"book, plutarch, read, studied, volume, volumes...",1.0,Narrative Reconstruction
5,26,reading,4,"book, volume",1.0,Narrative Reconstruction
0,10,walton_letters,0,,0.0,NaN
1,15,walton_letters,0,,1.0,Narrative Reconstruction
4,16,walton_letters,0,,1.0,Narrative Reconstruction
28,214,writing,3,"letters, notes",1.0,Narrative Reconstruction


## 5. Build Batch 03

Targets are filled in order. A passage picked for one target isn't picked again for another,
but `also_matches` shows every target it fits. Then a few random unlabelled passages are added.

In [6]:
picked, batch_rows = set(), []
for t in TARGETS:
    cand = scores[(scores["target"] == t) & scores["eligible"]]
    cand = cand[~cand["passage_id"].isin(labelled_ids | picked)]
    cand = cand.sort_values("score", ascending=False).head(QUOTA[t])
    for _, c in cand.iterrows():
        picked.add(c["passage_id"])
        batch_rows.append({"passage_id": c["passage_id"], "target": t,
                           "selection_method": TARGETS[t]["selection_method"],
                           "cue_score": c["score"], "matched_cues": c["matched_cues"]})

rng = np.random.default_rng(SEED)
pool = sorted(set(passages["passage_id"]) - labelled_ids - picked)
for pid in rng.choice(pool, size=min(N_RANDOM, len(pool)), replace=False):
    picked.add(int(pid))
    batch_rows.append({"passage_id": int(pid), "target": "random", "selection_method": "random",
                       "cue_score": 0, "matched_cues": ""})

batch = pd.DataFrame(batch_rows)

# Which other targets does each passage also match?
elig = scores[scores["eligible"]]
also = elig.groupby("passage_id")["target"].apply(list).to_dict()
batch["also_matches"] = [", ".join(x for x in also.get(pid, []) if x != t)
                         for pid, t in zip(batch["passage_id"], batch["target"])]

batch = batch.merge(passages[["passage_id", "chapter", "position", "text"]], on="passage_id")

# Empty annotation columns, ready to fill in
for col in ["therapeutic_label", "mechanism", "secondary_mechanism", "intensity",
            "annotator_confidence", "annotator_notes"]:
    batch[col] = ""

batch = batch[["passage_id", "chapter", "position", "target", "selection_method", "cue_score",
               "matched_cues", "also_matches", "text", "therapeutic_label", "mechanism",
               "secondary_mechanism", "intensity", "annotator_confidence", "annotator_notes"]]

print("Batch size:", len(batch))
print(batch["target"].value_counts().reindex(list(TARGETS) + ["random"]).fillna(0).astype(int).to_string())
batch[["passage_id", "chapter", "target", "selection_method", "cue_score", "matched_cues", "also_matches"]]

Batch size: 40
target
music                 6
poetry                6
walton_letters        4
writing               6
reading               4
support               5
reflection_despair    5
random                4


,passage_id,chapter,target,selection_method,cue_score,matched_cues,also_matches
0,109,Chapter 13,music,targeted,8,"guitar, music, musical",
1,111,Chapter 13,music,targeted,8,"guitar, music, sang",
2,102,Chapter 12,music,targeted,5,"harmony, instrument, songs",
3,108,Chapter 12,music,targeted,4,"music, sang","support, reflection_despair"
4,126,Chapter 15,music,targeted,3,"guitar, instrument",
5,152,Chapter 18,music,targeted,3,"harmony, song",
6,2,Letter 1,poetry,targeted,4,"poet, poets","walton_letters, reading"
7,7,Letter 2,poetry,targeted,4,"ancient mariner, poets",walton_letters
8,46,Chapter 5,poetry,targeted,4,"ancient mariner, coleridge",
9,5,Letter 2,poetry,targeted,2,poets,"walton_letters, reading, support"


## 6. Read the candidates (cue words highlighted)

Change `TARGET_TO_SHOW` to browse each group. Click a passage to expand the full text.

In [7]:
def highlight(text, target):
    out = escape(text)
    pats = compiled.get(target, {})
    for kind, colour in [("strong", "#ffe066"), ("weak", "#cde7ff")]:
        pat = pats.get(kind) if pats else None
        if pat is not None:
            out = pat.sub(lambda m: f"<mark style='background:{colour}'>{m.group(0)}</mark>", out)
    return out


def show_candidates(target):
    sub = batch[batch["target"] == target]
    html = [f"<h3>{target} ({len(sub)} passages)</h3>"]
    for _, r in sub.iterrows():
        html.append(
            f"<details style='margin-bottom:6px'><summary><b>Passage {r['passage_id']}</b> · "
            f"{r['chapter']} · cues: {escape(r['matched_cues']) or '—'}"
            f"{' · also: ' + escape(r['also_matches']) if r['also_matches'] else ''}</summary>"
            f"<div style='max-width:850px;line-height:1.5;padding:6px 0'>{highlight(r['text'], target)}</div>"
            f"</details>")
    display(HTML("".join(html)))


TARGET_TO_SHOW = "music"   # music, poetry, walton_letters, writing, reading, support, reflection_despair, random
show_candidates(TARGET_TO_SHOW)

## 7. Save

In [8]:
OUT = "frankenstein_batch03_candidates.csv"
batch.to_csv(OUT, index=False)

Path("results").mkdir(exist_ok=True)
scores.to_csv("results/target_cue_scores_all_passages.csv", index=False)  # scores only, no text

print("Saved", OUT, "and results/target_cue_scores_all_passages.csv")
if IN_COLAB:
    from google.colab import files
    files.download(OUT)

Saved frankenstein_batch03_candidates.csv and results/target_cue_scores_all_passages.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import pandas as pd

# ---- Load candidates ----
df = pd.read_csv("frankenstein_batch03_candidates.csv")

# ---- Annotations: passage_id -> (label, mechanism, secondary, intensity, confidence, notes) ----
annotations = {
    # ---------------- MUSIC ----------------
    109: (0, "", "", 0, "sure",
          "Music scene with cottagers; creature observes Felix's melancholy and Safie's arrival. "
          "No therapeutic processing—music is aesthetic backdrop, not a medium for emotional processing."),
    111: (0, "", "", 0, "unsure",
          "Safie's music draws tears from creature, and joy replaces sadness in cottagers. "
          "Aesthetic emotional response, but no reflective processing or meaning-making. "
          "Borderline—could be seen as music-mediated catharsis, but no explicit therapeutic work."),
    102: (0, "", "", 0, "sure",
          "Old man plays instrument; creature observes family. Descriptive observation, no processing."),
    108: (0, "", "", 0, "unsure",
          "Creature's thoughts become active, hopes to restore happiness, applies to learning language. "
          "Optimism and goal-setting, but no scriptotherapy or reflective processing."),
    126: (0, "", "", 0, "sure",
          "Old man plays guitar and reflects; creature prepares to approach cottage. No therapeutic processing."),
    152: (0, "", "", 0, "sure",
          "Rhine journey; Victor feels momentary tranquillity from nature and song of labourers. "
          "Nature-induced calm, not scriptotherapy."),

    # ---------------- POETRY ----------------
    2:   (1, "Narrative Reconstruction", "", 2, "unsure",
          "Walton reflects on his past reading and poetic ambitions; 'These reflections have dispelled "
          "the agitation with which I began my letter.' Writing/reflection provides emotional regulation. "
          "Mild but explicit therapeutic outcome."),
    7:   (0, "", "", 0, "sure",
          "Walton describes mixed sensations about voyage; literary allusion to Ancient Mariner. "
          "Emotional expression but not processing."),
    46:  (0, "", "", 0, "sure",
          "Victor's horror after creation; quotes Coleridge. Pure trauma expression, no processing."),
    5:   (0, "", "", 0, "sure",
          "Walton longs for a friend; explicitly says committing thoughts to paper is 'a poor medium.' "
          "Writing is named as insufficient, not therapeutic."),
    23:  (0, "", "", 0, "sure",
          "Childhood description of Elizabeth and Victor. No therapeutic processing."),
    56:  (0, "", "", 0, "unsure",
          "Victor finds 'consolation' in orientalist works while fleeing reflection. "
          "Consolation through reading, but framed as avoidance rather than processing."),

    # ---------------- WALTON LETTERS ----------------
    8:   (0, "", "", 0, "sure",
          "Walton asks sister to write to support his spirits. Letters as support, but no processing shown."),
    13:  (0, "", "", 0, "unsure",
          "Walton's journal records stranger's grief; Walton feels sympathy and brotherly love. "
          "Journaling as record-keeping, not emotional processing."),
    1:   (0, "", "", 0, "sure",
          "Opening letter and table of contents; Walton's excited description of voyage. No therapeutic processing."),
    9:   (0, "", "", 0, "sure",
          "Walton records strange accident in letter. Narration without processing."),

    # ---------------- WRITING ----------------
    51:  (0, "", "", 0, "sure",
          "Clerval urges Victor to write home; Victor trembles. Writing framed as obligation, not therapy."),
    54:  (1, "Emotional Catharsis", "", 2, "sure",
          "Elizabeth: 'I have written myself into better spirits, dear cousin.' "
          "Explicit scriptotherapy—writing improves mood. Clear therapeutic outcome."),
    124: (0, "", "", 0, "unsure",
          "Creature reads Victor's journal; 'I sickened as I read.' Reading causes distress, not therapy. "
          "Reflections follow but lead to despair, not resolution."),
    17:  (0, "", "", 0, "unsure",
          "Walton resolves to record Victor's narrative. Anticipation of writing, not processing itself."),
    117: (0, "", "", 0, "sure",
          "Safie's letters warm Felix's zeal. Letters as emotional support, but no processing shown."),
    55:  (0, "", "", 0, "sure",
          "Victor writes to Elizabeth to relieve her anxiety. Writing for others, fatiguing, not therapeutic for self."),

    # ---------------- READING ----------------
    122: (1, "Narrative Reconstruction", "", 3, "sure",
          "Creature reads Werter, Paradise Lost, Plutarch; applies reading to himself: "
          "'Who was I? What was I?' Reading as medium for self-reflection and meaning-making. "
          "Clear narrative reconstruction through reading."),
    27:  (0, "", "", 0, "sure",
          "Victor reads Agrippa, Paracelsus; intellectual curiosity, not therapeutic."),
    121: (0, "", "", 0, "sure",
          "Creature finds books; delight in possession. Setup for later reading, not processing itself."),
    106: (0, "", "", 0, "sure",
          "Felix reads aloud; creature observes and wants to learn language. Learning, not therapy."),

    # ---------------- SUPPORT (hard negatives) ----------------
    61:  (0, "", "", 0, "sure",
          "Hard negative: Clerval offers consolation and sympathy; Victor despairs. "
          "Support without catharsis. Matches pilot 67."),
    216: (0, "", "", 0, "sure",
          "Hard negative: Walton wants to reconcile Victor to life; Victor refuses. "
          "Support rejected, no processing."),
    180: (0, "", "", 0, "sure",
          "Hard negative: Kirwin offers comfort; Victor rejects. Support without processing."),
    144: (0, "", "", 0, "sure",
          "Hard negative: Victor pities creature, wants to console, but cannot sympathize. "
          "Ambivalent support, no catharsis."),
    179: (0, "", "", 0, "sure",
          "Hard negative: Kirwin shows kindness; Victor remains in despair. "
          "Support received but no therapeutic outcome."),

    # ---------------- REFLECTION + DESPAIR (hard negatives) ----------------
    166: (0, "", "", 0, "sure",
          "Hard negative: Creature's rage and threats; reflection + despair without resolution. Matches rule."),
    201: (0, "", "", 0, "sure",
          "Hard negative: Victor's father dies; Victor reflects on miseries, turns to revenge. "
          "Reflection + despair, no resolution."),
    65:  (0, "", "", 0, "sure",
          "Hard negative: Victor reflects on events, resolves to remain silent. "
          "Reflection + despair, no resolution."),
    91:  (0, "", "", 0, "sure",
          "Hard negative: Victor and creature confrontation; creature's misery. "
          "Reflection + despair without resolution."),
    225: (0, "", "", 0, "sure",
          "Hard negative: Creature's remorse and self-abhorrence. Reflection + despair, no resolution."),

    # ---------------- RANDOM ----------------
    104: (0, "", "", 0, "sure",
          "Random: Cottagers' poverty and creature's secret help. Descriptive, no processing."),
    28:  (0, "", "", 0, "sure",
          "Random: Victor's scientific studies and ambitions. Descriptive, no processing."),
    156: (0, "", "", 0, "sure",
          "Random: Travel narrative, Oxford. Descriptive, no processing."),
    176: (0, "", "", 0, "sure",
          "Random: Victor sees Clerval's corpse, convulsions. Pure trauma, no processing."),
}

# ---- Apply ----
for pid, (label, mech, sec, intensity, conf, notes) in annotations.items():
    mask = df["passage_id"] == pid
    df.loc[mask, "therapeutic_label"] = label
    df.loc[mask, "mechanism"] = mech
    df.loc[mask, "secondary_mechanism"] = sec
    df.loc[mask, "intensity"] = intensity
    df.loc[mask, "annotator_confidence"] = conf
    df.loc[mask, "annotator_notes"] = notes

# ---- Save ----
df.to_csv("frankenstein_batch03_labelled.csv", index=False)

# ---- Summary ----
print("Saved frankenstein_batch03_labelled.csv")
print("\nLabel distribution:")
print(df["therapeutic_label"].value_counts().to_string())
print("\nMechanism distribution (positives only):")
print(df[df["therapeutic_label"] == 1]["mechanism"].value_counts().to_string())
print("\nPositives by target:")
print(df[df["therapeutic_label"] == 1][["passage_id", "target", "mechanism"]].to_string(index=False))

Saved frankenstein_batch03_labelled.csv

Label distribution:
therapeutic_label
0.0    37
1.0     3

Mechanism distribution (positives only):
mechanism
Narrative Reconstruction    2
Emotional Catharsis         1

Positives by target:
 passage_id  target                mechanism
          2  poetry Narrative Reconstruction
         54 writing      Emotional Catharsis
        122 reading Narrative Reconstruction


/tmp/ipykernel_1391/142109128.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "mechanism"] = mech
/tmp/ipykernel_1391/142109128.py:132: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "secondary_mechanism"] = sec
/tmp/ipykernel_1391/142109128.py:134: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'sure' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, "annotator_confidence"] = conf
/tmp/ipykernel_1391/142109128.py:135: FutureWarning: Setting an item of incompatible dtype is dep

## 8. How to label Batch 03

1. Open `frankenstein_batch03_candidates.csv` in Excel or Google Sheets.
2. Read each passage fully. Ignore the highlighted words while deciding; they're only there to find passages.
3. Fill in `therapeutic_label`, `mechanism`, `secondary_mechanism`, `intensity`, `annotator_confidence` (sure / unsure) and `annotator_notes`, **using guidelines v1.1**.
4. Don't change `selection_method`. It records *how* the passage was found, not what it is.
5. Save as `frankenstein_batch03_labelled.csv` and run Section 9 to merge.

✍️ **Write here:** which target produced the most false positives, and why?

## 9. (After labelling) Merge into dataset v3

Set `MERGE_NOW = True` only after you've labelled Batch 03. The original 39 passages get `selection_method = "pilot"`.

In [12]:
MERGE_NOW = True  # set to True after you've labelled Batch 03
LABELLED_FILE = "frankenstein_batch03_labelled.csv"

if MERGE_NOW:
    new = pd.read_csv(find_file(LABELLED_FILE))
    new = new.dropna(subset=["therapeutic_label"])

    old = labelled.copy()
    old["selection_method"] = "pilot"
    old["target"] = ""
    for col in ["secondary_mechanism", "annotator_confidence"]:
        if col not in old:
            old[col] = ""

    cols = ["passage_id", "text", "therapeutic_label", "mechanism", "secondary_mechanism",
            "intensity", "annotator_confidence", "annotator_notes", "selection_method", "target"]
    v3 = pd.concat([old[cols], new[cols]], ignore_index=True)
    v3 = v3.drop_duplicates("passage_id", keep="last").sort_values("passage_id")
    v3.to_csv("frankenstein_training_dataset_v3.csv", index=False)

    print("v3 size:", len(v3))
    print(v3["therapeutic_label"].value_counts().to_string())
    print(v3["mechanism"].value_counts().to_string())
else:
    print("Skipped. Set MERGE_NOW = True after labelling Batch 03.")

v3 size: 79
therapeutic_label
0.0    57
1.0    22
mechanism
Narrative Reconstruction    15
Emotional Catharsis          7


In [13]:
from google.colab import files
files.download("frankenstein_batch03_labelled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>